# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print description
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nPublished: {metadata.datePublished}\nIdentifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Entities in the Croissant dataset are referenced by their `@id` fields for unambiguous access. Here, we enumerate the record sets and list their field and column `@id`s.

In [ ]:
# List all record sets and their field/column @id references
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
rs_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', 'N/A')} (@id: {rs.id})")
    rs_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {getattr(f, 'name', 'N/A')} (@id: {f.id})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for c in rs.columns:
            print(f"    - {getattr(c, 'name', 'N/A')} (@id: {c.id})")
    print()
# If there are no record sets, let the user know
if not rs_ids:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use record set and field `@id`s from the overview.

In [ ]:
# If there are record sets, load their records into DataFrames
dataframes = {}

for rs_id in rs_ids:
    print(f"Loading records from RecordSet '@id': {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records with fields: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"No records found in record set {rs_id}.")
    print("-"*60)
if not dataframes:
    print("No tabular data extracted from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below, we'll demonstrate EDA on the largest available record set. Choose field (column) `@id`s for numeric operations as shown in the Data Extraction step above.

In [ ]:
# Pick one record set with tabular data for demonstration
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    print(f"Proceeding with RecordSet '@id': {first_rs_id}")
    
    # Choose a numeric column to filter, e.g., any with 'count', 'mw', 'coverage', or similar in the name
    sample_numeric_candidates = [col for col in df.columns if any(sub in col.lower() for sub in ['count','mw','coverage','abundance','score','number','peptide','sum','norm'])]
    if sample_numeric_candidates:
        numeric_field = sample_numeric_candidates[0]  # Choose the first found candidate
        print(f"Chosen numeric field for EDA: {numeric_field}")
        # Coerce to float
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > median ({threshold}): {len(filtered_df)} records\n")
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))
        # Choose a grouping column, e.g., any with 'type','category','group','class' in name
        group_candidates = [col for col in df.columns if any(sub in col.lower() for sub in ['type','category','group','sample','condition','class']) and col != numeric_field]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head(5))
        else:
            group_field = None
            print("No suitable group field found for grouping.")
    else:
        print("No suitable numeric fields found for EDA in this record set.")
        numeric_field = None
        group_field = None
else:
    print("No record set DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # Optionally, visualize normalized values as well
    if f"{numeric_field}_normalized" in filtered_df:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=30, color='orange')
        plt.title(f'Normalized {numeric_field} (filtered)')
        plt.xlabel(f'{numeric_field}_normalized')
        plt.ylabel('Count')
        plt.show()
    # If grouped data, plot mean per group
    if group_field:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No suitable numeric field or record set found for visualization.')

## 6. Conclusion
This notebook demonstrated how to load, explore, and process a Croissant-structured FAIR² dataset using the `mlcroissant` library.

- Dataset entities are referenced by their canonical `@id`s, facilitating reproducible access.
- Record sets, fields, and columns are enumerated, and tabular data is loaded dynamically.
- Basic EDA and visualization steps were applied to example numeric fields, with normalization and group summaries.

You may extend this template for further domain-specific analysis or advanced machine learning workflows. For more information on the dataset, refer to its [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and [project page](https://sen.science/doi/10.71728/senscience.vq4a-28xa).